# reconcile_gst2b_env — T4 dry-run

20-step GRPO + LoRA dry-run on Colab T4 (16 GB). Target: finish in ≤25 min,
produce `data/dryrun/training_curves.json` with ≥4 eval points, save a
LoRA checkpoint that loads via `AutoModelForCausalLM.from_pretrained`.

If OOM: script auto-downgrades to Qwen2.5-1.5B-Instruct + group_size=4 and
notes the fallback in `training_curves.json → config.fallback = true`.

## 1 — Install deps

In [ ]:
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'xformers<0.0.27' 'trl<0.9.0' peft accelerate bitsandbytes
!pip install -q networkx pandas numpy matplotlib pydantic>=2.5

## 2 — Clone repo and install env package

In [ ]:
import os, sys
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/meta-pytorch/OpenEnv.git')
BRANCH = os.environ.get('REPO_BRANCH', 'scaffold/reconcile-gst2b')
!git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/OpenEnv || true
%cd /content/OpenEnv
sys.path.insert(0, '/content/OpenEnv')
sys.path.insert(0, '/content/OpenEnv/src')

## 3 — 20-step dry-run

In [ ]:
!python -m envs.reconcile_gst2b_env.scripts.training --tier=t4 --steps=20 --output-dir=data/dryrun --eval-every=5

## 4 — Plot training curves

**Scaffold run — flat curves are expected; the GRPO step is a placeholder. Real training loop swaps in Cell 3 for the pitch demo.**

In [ ]:
import json
import matplotlib.pyplot as plt
with open('data/dryrun/training_curves.json') as f:
    curves = json.load(f)
steps = curves['steps']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for k, style in [('R1', '-o'), ('R2', '-s'), ('R3', '-^'), ('R4', '-d')]:
    axes[0].plot(steps, curves[k], style, label=k)
axes[0].set_title('per-component reward (eval seeds 9030..9049)')
axes[0].set_xlabel('training step')
axes[0].set_ylabel('component mean')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(steps, curves['total'], '-o', color='black')
axes[1].set_title('total reward (composite)')
axes[1].set_xlabel('training step')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('data/dryrun/training_curves.png', dpi=120)
plt.show()
print(f"config: {curves['config']}")
print(f"n eval points: {len(steps)}  (done-gate requires >=4)")

## 5 — Sanity-check: reload checkpoint and score one eval seed

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import glob, os

ckpts = sorted(glob.glob('data/dryrun/checkpoints/step_*'))
assert ckpts, 'no checkpoints saved — training failed before first ckpt'
ckpt = ckpts[-1]
print(f'loading {ckpt}')

# Load base + LoRA adapter.
base_name = curves['config']['model']
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForCausalLM.from_pretrained(base_name, torch_dtype='auto', device_map='auto')
try:
    model = PeftModel.from_pretrained(model, ckpt)
    print('LoRA adapter loaded')
except Exception as exc:
    print(f'checkpoint was saved without adapter (full-model save): {exc}')

# Single-seed eval on a trained checkpoint.
from envs.reconcile_gst2b_env.scripts.training import _stub_eval_policy_factory, evaluate_checkpoint
policy = _stub_eval_policy_factory(model, tokenizer)
trajectory = policy(9030)
print(f'trained-policy trajectory on seed 9030: {len(trajectory)} actions')

## 6 — Append results to baseline_metrics for the 'trained' slot

In [ ]:
means = evaluate_checkpoint(policy)
trained_summary = {
    'condition': 'trained',
    'execution_mode': 'real_dryrun',
    'n_rollouts': len(curves['config']['eval_seeds']),
    'total_mean': round(means['total'], 4),
    'component_means': {k: round(means[k], 4) for k in ('R1','R2','R3','R4')},
    'checkpoint': ckpts[-1],
    'note': 'dry-run (20 training steps); not a real trained baseline.',
}
import json
os.makedirs('data', exist_ok=True)
with open('data/baseline_metrics_trained_dryrun.json', 'w') as f:
    json.dump(trained_summary, f, indent=2)
print(json.dumps(trained_summary, indent=2))